#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType,DateType
from pyspark.sql.functions import trim,col,length

#Reading From Bronze Layer

In [0]:
df=spark.table("workspace.bronze.crm_sales_details")

#Data Transformation

#Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

#Normalization

## Sales and Price Corrections

In [0]:
df =(df.withColumn(
    "sls_price",
    F.when(
            (col('sls_price').isNull())|(col('sls_price')<=0),
            F.when(
                col('sls_quantity')!=0,
                col('sls_sales')/col('sls_quantity')
            ).otherwise(None)
        ).otherwise(col('sls_price'))
     )
) 


## Date casting

In [0]:
df = (
    df
    .withColumn(
    "sls_order_dt",
    F.when((col('sls_order_dt')==0)|(length(col('sls_order_dt')) != 8),None)
    .otherwise(F.to_date(col('sls_order_dt').cast('string'), 'yyyyMMdd')))

    .withColumn(
    "sls_ship_dt",
    F.when((col('sls_ship_dt')==0)|(length(col('sls_ship_dt')) != 8),None)
    .otherwise(F.to_date(col('sls_ship_dt').cast('string'), 'yyyyMMdd')))

    .withColumn(
    "sls_due_dt",
    F.when((col('sls_due_dt')==0)|(length(col('sls_due_dt')) != 8),None)
    .otherwise(F.to_date(col('sls_due_dt').cast('string'), 'yyyyMMdd')))

)


#Renaming columns

In [0]:
rename_cols = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old_name, new_name in rename_cols.items():
    df = df.withColumnRenamed(old_name, new_name)


## Sanity checks of dataframe

In [0]:
display(df.limit(10))

#Write to Silver Layer

In [0]:
(
df.write
.mode("overwrite")
.format('delta')
.saveAsTable("silver.crm_sales")
)